# Testing Suit

In [1]:
# Import library that allow to work with quantum circuits
from qiskit import qpy
import evaluation_functions as evf
import mcts_group as mcts
from structure import Circuit
import itertools
import os
# import seaborn as sns
from tqdm import tqdm
import csv

In [6]:
num_runs = 10
params = {
    'collect_data': [True],
    'evaluation_function': [evf.h2, evf.lih, evf.h2o, evf.vqls_0, evf.vqls_1],
    # 'choices': [{'a': 20, 'd': 20, 's': 20, 'c': 20, 'p': 20}],
    # 'stop_deterministic': [False],

# Change from here
    'group_by_gates': [False],
    'group_by_action': [True],
    'terminal_nodes': [False],
    'group_by_change': [False],
    'group_by_swap': [False],
    'finite_progressive_widening': [True],
}
name = 'HERE_USE_ONLY_FROM_BELOW'
# name = "standard_nc"
# name = "terminal_nc"
# name = "actions_pw"
# name = "gates_pw"
# name = "actions_gates_pw"
# name = "gates_pw_terminal"
# name = "actions_pw_terminal"
# name = "actions_gates_pw_terminal"
# name = "actions_change_gates_pw_swap"
# name = "actions_change_gates_pw_swap_terminal"
# name = "standard"
# name = "terminal"
# name = "actions"
# name = "gates"
# name = "actions_gates"
# name = "gates_terminal"
# name = "actions_terminal"
# name = "actions_gates_terminal"
# name = "actions_change_gates_swap"
# name = "actions_change_gates_swap_terminal"
# Till here

In [ ]:
nqbits = {'h2': 4, 'lih': 10, 'h2o': 8,'vqls_1': 4,'vqls_0': 4}
budgets = {'h2': 5000, 'lih': 5000, 'h2o': 5000,'vqls_1': 5000,'vqls_0': 5000}
keys = list(params)
results_folder = 'results'

def test_all_vars_one_problem(max_depth, verbose=False):    
    for values in tqdm(itertools.product(*map(params.get,keys))):
        print(values)
        for i in range(num_runs):
            evf = values[1]
            evf_name = evf.__name__
            folder_name = f"{results_folder}/{name}/{evf_name}"
            os.makedirs(folder_name, exist_ok=True)  # Create the folder if it doesn't exist
            filename = f"{name}_{i}.csv"
            csv_path = os.path.join(folder_name, filename)
            if not os.path.isfile(csv_path):
                root = mcts.Node(Circuit(variable_qubits=nqbits[evf_name], ancilla_qubits=0), max_depth=max_depth)
                results = mcts.mcts(root, **dict(zip(keys,values)), budget=budgets[evf_name], verbose=verbose)
                objective_values = results['data'] 
                objective_values.to_csv(csv_path, index=False)

                qc_last = results['qc'][-1]
                qc_best = results['best_qc']
                data = [
                    ["Category", "EnergyBefore", "ValueBefore", "EnergyAfter", "ValueAfter", "H", "Cx", "Rx", "Ry", "Rz"]
                ]
                for id, circ in {"last": qc_last, "best": qc_best}.items():
                    arr = []
                    with open(f"{results_folder}/{name}/{evf_name}/{name}_{i}_{id}_circuit.qpy", "wb") as file:
                        qpy.dump(circ, file)
                    optim = [evf(quantum_circuit=circ)] if "vqls_0" in evf_name else evf(quantum_circuit=circ, gradient=True)
                    energy_before = optim[0]
                    value_before = -energy_before    
                    energy_after = optim[-1]
                    value_after = -energy_after    
                    gate_counts = dict(circ.count_ops())
                    h_gates = gate_counts["h"] if "h" in gate_counts.keys() else 0
                    cx_gates = gate_counts["rx"] if "rx" in gate_counts.keys() else 0
                    rx_gates = gate_counts["ry"] if "ry" in gate_counts.keys() else 0
                    ry_gates = gate_counts["rz"] if "rz" in gate_counts.keys() else 0
                    rz_gates = gate_counts["cx"] if "cx" in gate_counts.keys() else 0
                    arr = [id, energy_before, value_before, energy_after, value_after, h_gates, cx_gates, rx_gates, ry_gates, rz_gates]
                    data.append(arr)

                with open(f"{results_folder}/{name}/{evf_name}/{name}_{i}_optimized.csv", mode='w', newline='', encoding='utf-8') as file:
                    writer = csv.writer(file)
                    writer.writerows(data)
        
test_all_vars_one_problem(10, False)

0it [00:00, ?it/s]

(True, <function h2 at 0x10ff689a0>, False, True, False, False, False, True)
Step = 0,  Energy = -1.03529379 Ha
Step = 2,  Energy = -1.04241504 Ha
Step = 4,  Energy = -1.04920350 Ha
Step = 6,  Energy = -1.05564563 Ha
Step = 8,  Energy = -1.06172987 Ha
Step = 10,  Energy = -1.06744694 Ha
Step = 12,  Energy = -1.07279014 Ha
Step = 14,  Energy = -1.07775566 Ha
Step = 16,  Energy = -1.08234278 Ha
Step = 18,  Energy = -1.08655402 Ha
Step = 20,  Energy = -1.09039522 Ha
Step = 22,  Energy = -1.09387555 Ha
Step = 24,  Energy = -1.09700737 Ha
Step = 26,  Energy = -1.09980607 Ha
Step = 28,  Energy = -1.10228971 Ha
Step = 30,  Energy = -1.10447861 Ha
Step = 32,  Energy = -1.10639490 Ha
Step = 34,  Energy = -1.10806190 Ha
Step = 36,  Energy = -1.10950353 Ha
Step = 38,  Energy = -1.11074370 Ha
Step = 40,  Energy = -1.11180574 Ha
Step = 42,  Energy = -1.11271184 Ha
Step = 44,  Energy = -1.11348267 Ha
Step = 46,  Energy = -1.11413703 Ha
Step = 48,  Energy = -1.11469168 Ha
Step = 50,  Energy = -1.1151

1it [05:39, 339.47s/it]

Step = 50,  Energy = -1.11734880 Ha
Step = 52,  Energy = -1.11734894 Ha
Step = 54,  Energy = -1.11734855 Ha
Step = 56,  Energy = -1.11734887 Ha
Step = 58,  Energy = -1.11734899 Ha
Step = 60,  Energy = -1.11734879 Ha
Step = 62,  Energy = -1.11734892 Ha
Step = 64,  Energy = -1.11734902 Ha
Step = 66,  Energy = -1.11734890 Ha
Landscape is flat
(True, <function lih at 0x10ff69120>, False, True, False, False, False, True)
Landscape is flat
Landscape is flat
Landscape is flat
Landscape is flat


/opt/homebrew/lib/python3.11/site-packages/autograd/tracer.py:14: UserWarning: Output seems independent of input.
  warnings.warn("Output seems independent of input.")


...

Landscape is flat
(True, <function h2o at 0x175333e20>, False, True, False, False, False, True)
Step = 0,  Energy = -75.02553200 Ha
Landscape is flat
Step = 0,  Energy = -75.20944886 Ha
Step = 2,  Energy = -75.20998817 Ha
Step = 4,  Energy = -75.21051620 Ha
Step = 6,  Energy = -75.21103241 Ha
Step = 8,  Energy = -75.21153630 Ha
Step = 10,  Energy = -75.21202739 Ha
Step = 12,  Energy = -75.21250524 Ha
Step = 14,  Energy = -75.21296944 Ha
Step = 16,  Energy = -75.21341963 Ha
Step = 18,  Energy = -75.21385549 Ha
Step = 20,  Energy = -75.21427676 Ha
Step = 22,  Energy = -75.21468322 Ha
Step = 24,  Energy = -75.21507470 Ha
Step = 26,  Energy = -75.21545108 Ha
Step = 28,  Energy = -75.21581231 Ha
Step = 30,  Energy = -75.21615836 Ha
Step = 32,  Energy = -75.21648927 Ha
Step = 34,  Energy = -75.21680512 Ha
Step = 36,  Energy = -75.21710604 Ha
Step = 38,  Energy = -75.21739218 Ha
Step = 40,  Energy = -75.21766377 Ha
Step = 42,  Energy = -75.21792103 Ha
Step = 44,  Energy = -75.21816426 Ha
Step

3it [12:08, 217.86s/it]

Step = 68,  Energy = -75.11565772 Ha
Step = 70,  Energy = -75.11565732 Ha
Step = 72,  Energy = -75.11565683 Ha
Step = 74,  Energy = -75.11565646 Ha
Step = 76,  Energy = -75.11565630 Ha
Landscape is flat
(True, <function vqls_0 at 0x175332ac0>, False, True, False, False, False, True)


4it [44:21, 895.30s/it]

(True, <function vqls_1 at 0x175333420>, False, True, False, False, False, True)

Final value of the cost function is  0.0216 
Landscape is flat

Final value of the cost function is  0.0000 

Final value of the cost function is  0.0185 

Final value of the cost function is  0.0000 

Final value of the cost function is  0.0136 
Landscape is flat

Final value of the cost function is  0.0135 
Landscape is flat

Final value of the cost function is  0.0135 
Landscape is flat

Final value of the cost function is  0.0000 

Final value of the cost function is  0.0222 
Landscape is flat

Final value of the cost function is  0.0135 

Final value of the cost function is  0.0347 

Final value of the cost function is  0.0063 
